In [1]:
%matplotlib inline
import openmc
import math
import openmc.deplete
import numpy as np
import openmc.mgxs as mgxs

material specification

In [2]:
# 13.5% enriched UO2
inner_fuel = openmc.Material(name="inner UO2",material_id=1)
inner_fuel.temperature = 900.0
inner_fuel.add_nuclide("O16",4.65353E-02)
inner_fuel.add_nuclide("U235",3.14113E-03)
inner_fuel.add_nuclide("U238",2.01265E-02)
inner_fuel.set_density("atom/b-cm",6.98029E-02)
inner_fuel.volume = 112166.502046326

# 16.5% enriched UO2
middle_fuel = openmc.Material(name="middle UO2",material_id=2)
middle_fuel.temperature = 900.0
middle_fuel.add_nuclide("O16",4.65508E-02)
middle_fuel.add_nuclide("U235",3.84044E-03)
middle_fuel.add_nuclide("U238",1.94350E-02)
middle_fuel.set_density("atom/b-cm",6.98262E-02)
middle_fuel.volume = 192285.432079416 

# 18.5% enriched UO2
outer_fuel = openmc.Material(name="outer UO2",material_id=3)
outer_fuel.temperature = 900.0
outer_fuel.add_nuclide("O16",4.65612E-02)
outer_fuel.add_nuclide("U235",4.30691E-03)
outer_fuel.add_nuclide("U238",1.89737E-02)
outer_fuel.set_density("atom/b-cm",6.98418E-02)
outer_fuel.volume = 288428.148119124

# 0.1Mpa helium gas fill gas plenum
helium = openmc.Material(name="helium",material_id=4)
helium.temperature = 293.6
helium.add_element("He",2.4724E-05)
helium.set_density("atom/b-cm",2.4724E-05)

# T91 steel
t91 = openmc.Material(name="T91 steel",material_id=5)
t91.temperature = 600
t91.add_element("C",3.8900E-04)
t91.add_element("Cr",7.86904E-03)
t91.add_element("Ni",1.59481E-04)
t91.add_element("Mn",3.8300E-04)
t91.add_element("Mo",4.62700E-04)
t91.add_element("Si",5.81562E-04)
t91.add_element("Nb",4.0200E-05)
t91.add_element("P",3.0200E-05)
t91.add_element("N",1.66570E-04)
t91.add_element("O",7.28445E-06)
t91.add_element("Cu",7.36000E-05)
t91.add_element("V",1.97003E-04)
t91.add_element("Al",6.9300E-05)
t91.add_element("Fe",7.42322E-02)
t91.set_density("atom/b-cm",8.46611E-02)

# YZrO as axial reflector
yzro = openmc.Material(name="YZrO reflector",material_id=6)
yzro.temperature = 600
yzro.add_element("O",5.8100E-02)
yzro.add_element("Zr",2.78078E-02)
yzro.add_element("Y",1.6300E-03)
yzro.set_density("atom/b-cm",8.7538E-02)

# Pb-Bi coolant
coolant = openmc.Material(name="Pb-Bi coolant",material_id=7)
coolant.temperature = 600
coolant.add_element("Pb",1.31669E-02)
coolant.add_element("Bi",1.66170E-02)
coolant.set_density("atom/b-cm",2.9784E-02)

# 40pct enriched B4C as CR
cr_40pct_b4c = openmc.Material(name="cr_40pct_B4C",material_id=8)
cr_40pct_b4c.temperature = 600
cr_40pct_b4c.add_nuclide("B10",3.18706E-02)
cr_40pct_b4c.add_nuclide("B11",4.78059E-02)
cr_40pct_b4c.add_nuclide("C12",1.99191E-02)
cr_40pct_b4c.set_density("atom/b-cm",9.95956E-02)

# 60pct enriched B4C as CR
cr_60pct_b4c = openmc.Material(name="cr_60pct_B4C",material_id=9)
cr_60pct_b4c.temperature = 600
cr_60pct_b4c.add_nuclide("B10",4.85194E-02)
cr_60pct_b4c.add_nuclide("B11",3.23463E-02)
cr_60pct_b4c.add_nuclide("C12",2.02164E-02)
cr_60pct_b4c.set_density("atom/b-cm",1.01082E-01)

# 90pct enriched B4C as SR
sr_90pct_b4c = openmc.Material(name="sr85_B4C",material_id=10)
sr_90pct_b4c.temperature = 600
sr_90pct_b4c.add_nuclide("B10",7.44458E-02)
sr_90pct_b4c.add_nuclide("B11",8.27176E-03)
sr_90pct_b4c.add_nuclide("C12",2.06794E-02)
sr_90pct_b4c.set_density("atom/b-cm",1.03397E-01)

materials = openmc.Materials((inner_fuel,middle_fuel,outer_fuel,helium,cr_40pct_b4c,cr_60pct_b4c,sr_90pct_b4c,t91,yzro,coolant))
materials.export_to_xml()

Geometry specification of inner fuel pin

In [3]:
# basic surface of pin
r_pin_f = openmc.ZCylinder(r=0.535)
r_protect_gas_f = openmc.ZCylinder(r=0.55)
r_cladding_f = openmc.ZCylinder(r=0.6)
z_top_active_region_f = openmc.ZPlane(z0=90)
z_top_upper_reflector_f = openmc.ZPlane(z0=149.5)
z_top_upper_gas_plenum_f = openmc.ZPlane(z0=180)
z_top_upper_plug_f = openmc.ZPlane(z0=185.05)
z_bottom_active_region_f = openmc.ZPlane(z0=0)
z_bottom_lower_reflector_f = openmc.ZPlane(z0=-22.5)
z_bottom_lower_plug_f = openmc.ZPlane(z0=-27)

# inner pin universe
inner_active_pin_c = openmc.Cell(fill=inner_fuel,region=-r_pin_f&+z_bottom_active_region_f&-z_top_active_region_f)
inner_upper_reflector_c = openmc.Cell(fill=yzro,region=-r_pin_f&+z_top_active_region_f&-z_top_upper_reflector_f)
inner_lower_reflector_c = openmc.Cell(fill=yzro,region=-r_pin_f&+z_bottom_lower_reflector_f&-z_bottom_active_region_f)
inner_protect_gas_c = openmc.Cell(fill=helium,region=+r_pin_f&-r_protect_gas_f&+z_bottom_lower_reflector_f&-z_top_upper_reflector_f)
inner_upper_gas_plenum_c = openmc.Cell(fill=helium,region=-r_protect_gas_f&+z_top_upper_reflector_f&-z_top_upper_gas_plenum_f)
inner_cladding_c = openmc.Cell(fill=t91,region=+r_protect_gas_f&-r_cladding_f&+z_bottom_lower_reflector_f&-z_top_upper_gas_plenum_f)
inner_upper_plug_c = openmc.Cell(fill=t91,region=-r_cladding_f&+z_top_upper_gas_plenum_f&-z_top_upper_plug_f)
inner_lower_plug_c = openmc.Cell(fill=t91,region=-r_cladding_f&-z_bottom_lower_reflector_f&+z_bottom_lower_plug_f)
inner_coolant_c = openmc.Cell(fill=coolant,region=+r_cladding_f|-z_bottom_lower_plug_f|+z_top_upper_plug_f)

inner_pin_u = openmc.Universe(cells=[inner_active_pin_c,inner_upper_reflector_c,inner_lower_reflector_c,inner_protect_gas_c,inner_upper_gas_plenum_c,inner_cladding_c,inner_upper_plug_c,inner_lower_plug_c,inner_coolant_c])

# middle pin universe
middle_active_pin_c = openmc.Cell(fill=middle_fuel,region=-r_pin_f&+z_bottom_active_region_f&-z_top_active_region_f)
middle_upper_reflector_c = openmc.Cell(fill=yzro,region=-r_pin_f&+z_top_active_region_f&-z_top_upper_reflector_f)
middle_lower_reflector_c = openmc.Cell(fill=yzro,region=-r_pin_f&+z_bottom_lower_reflector_f&-z_bottom_active_region_f)
middle_protect_gas_c = openmc.Cell(fill=helium,region=+r_pin_f&-r_protect_gas_f&+z_bottom_lower_reflector_f&-z_top_upper_reflector_f)
middle_upper_gas_plenum_c = openmc.Cell(fill=helium,region=-r_protect_gas_f&+z_top_upper_reflector_f&-z_top_upper_gas_plenum_f)
middle_cladding_c = openmc.Cell(fill=t91,region=+r_protect_gas_f&-r_cladding_f&+z_bottom_lower_reflector_f&-z_top_upper_gas_plenum_f)
middle_upper_plug_c = openmc.Cell(fill=t91,region=-r_cladding_f&+z_top_upper_gas_plenum_f&-z_top_upper_plug_f)
middle_lower_plug_c = openmc.Cell(fill=t91,region=-r_cladding_f&-z_bottom_lower_reflector_f&+z_bottom_lower_plug_f)
middle_coolant_c = openmc.Cell(fill=coolant,region=+r_cladding_f|-z_bottom_lower_plug_f|+z_top_upper_plug_f)

middle_pin_u = openmc.Universe(cells=[middle_active_pin_c,middle_upper_reflector_c,middle_lower_reflector_c,middle_protect_gas_c,middle_upper_gas_plenum_c,middle_cladding_c,middle_upper_plug_c,middle_lower_plug_c,middle_coolant_c])

# outer pin universe
outer_active_pin_c = openmc.Cell(fill=outer_fuel,region=-r_pin_f&+z_bottom_active_region_f&-z_top_active_region_f)
outer_upper_reflector_c = openmc.Cell(fill=yzro,region=-r_pin_f&+z_top_active_region_f&-z_top_upper_reflector_f)
outer_lower_reflector_c = openmc.Cell(fill=yzro,region=-r_pin_f&+z_bottom_lower_reflector_f&-z_bottom_active_region_f)
outer_protect_gas_c = openmc.Cell(fill=helium,region=+r_pin_f&-r_protect_gas_f&+z_bottom_lower_reflector_f&-z_top_upper_reflector_f)
outer_upper_gas_plenum_c = openmc.Cell(fill=helium,region=-r_protect_gas_f&+z_top_upper_reflector_f&-z_top_upper_gas_plenum_f)
outer_cladding_c = openmc.Cell(fill=t91,region=+r_protect_gas_f&-r_cladding_f&+z_bottom_lower_reflector_f&-z_top_upper_gas_plenum_f)
outer_upper_plug_c = openmc.Cell(fill=t91,region=-r_cladding_f&+z_top_upper_gas_plenum_f&-z_top_upper_plug_f)
outer_lower_plug_c = openmc.Cell(fill=t91,region=-r_cladding_f&-z_bottom_lower_reflector_f&+z_bottom_lower_plug_f)
outer_coolant_c = openmc.Cell(fill=coolant,region=+r_cladding_f|-z_bottom_lower_plug_f|+z_top_upper_plug_f)

outer_pin_u = openmc.Universe(cells=[outer_active_pin_c,outer_upper_reflector_c,outer_lower_reflector_c,outer_protect_gas_c,outer_upper_gas_plenum_c,outer_cladding_c,outer_upper_plug_c,outer_lower_plug_c,outer_coolant_c])

# 40pct control rod universe down
r_rod_40pct_down_f = openmc.ZCylinder(r=2.0)
r_pipe_40pct_down_f = openmc.ZCylinder(r=2.2)
# z_bottom_absorber_40pct_down_f = openmc.ZPlane(z0=-1)
# z_top_absorber_40pct_down_f = openmc.ZPlane(z0=91)
# z_bottom_reflector_40pct_down_f = openmc.ZPlane(z0=-93)
z_bottom_absorber_40pct_down_f = openmc.ZPlane(z0=31)
z_top_absorber_40pct_down_f = openmc.ZPlane(z0=123)
z_bottom_reflector_40pct_down_f = openmc.ZPlane(z0=-61)

cr_absorber_40pct_down_c = openmc.Cell(fill=cr_40pct_b4c,region=-r_rod_40pct_down_f&+z_bottom_absorber_40pct_down_f&-z_top_absorber_40pct_down_f)
cr_reflector_40pct_down_c = openmc.Cell(fill=yzro,region=-r_rod_40pct_down_f&+z_bottom_reflector_40pct_down_f&-z_bottom_absorber_40pct_down_f)
cr_pipe_40pct_down_c = openmc.Cell(fill=t91,region=+r_rod_40pct_down_f&-r_pipe_40pct_down_f&+z_bottom_reflector_40pct_down_f&-z_top_absorber_40pct_down_f)
cr_coolant_40pct_down_c = openmc.Cell(fill=coolant,region=+r_pipe_40pct_down_f|-z_bottom_reflector_40pct_down_f|+z_top_absorber_40pct_down_f)

cr_40pct_down_u = openmc.Universe(cells=[cr_absorber_40pct_down_c,cr_reflector_40pct_down_c,cr_pipe_40pct_down_c,cr_coolant_40pct_down_c])

# 40pct control rod universe up
r_rod_40pct_up_f = openmc.ZCylinder(r=2.0)
r_pipe_40pct_up_f = openmc.ZCylinder(r=2.2)
z_bottom_absorber_40pct_up_f = openmc.ZPlane(z0=91)
z_top_absorber_40pct_up_f = openmc.ZPlane(z0=183)
z_bottom_reflector_40pct_up_f = openmc.ZPlane(z0=-1)

cr_absorber_40pct_up_c = openmc.Cell(fill=cr_40pct_b4c,region=-r_rod_40pct_up_f&+z_bottom_absorber_40pct_up_f&-z_top_absorber_40pct_up_f)
cr_reflector_40pct_up_c = openmc.Cell(fill=yzro,region=-r_rod_40pct_up_f&+z_bottom_reflector_40pct_up_f&-z_bottom_absorber_40pct_up_f)
cr_pipe_40pct_up_c = openmc.Cell(fill=t91,region=+r_rod_40pct_up_f&-r_pipe_40pct_up_f&+z_bottom_reflector_40pct_up_f&-z_top_absorber_40pct_up_f)
cr_coolant_40pct_up_c = openmc.Cell(fill=coolant,region=+r_pipe_40pct_up_f|-z_bottom_reflector_40pct_up_f|+z_top_absorber_40pct_up_f)

cr_40pct_up_u = openmc.Universe(cells=[cr_absorber_40pct_up_c,cr_reflector_40pct_up_c,cr_pipe_40pct_up_c,cr_coolant_40pct_up_c])

# 60pct control rod universe down
r_rod_60pct_down_f = openmc.ZCylinder(r=2.0)
r_pipe_60pct_down_f = openmc.ZCylinder(r=2.2)
# z_bottom_absorber_60pct_down_f = openmc.ZPlane(z0=-1)
# z_top_absorber_60pct_down_f = openmc.ZPlane(z0=91)
# z_bottom_reflector_60pct_down_f = openmc.ZPlane(z0=-93)
z_bottom_absorber_60pct_down_f = openmc.ZPlane(z0=31)
z_top_absorber_60pct_down_f = openmc.ZPlane(z0=123)
z_bottom_reflector_60pct_down_f = openmc.ZPlane(z0=-61)

cr_absorber_60pct_down_c = openmc.Cell(fill=cr_60pct_b4c,region=-r_rod_60pct_down_f&+z_bottom_absorber_60pct_down_f&-z_top_absorber_60pct_down_f)
cr_reflector_60pct_down_c = openmc.Cell(fill=yzro,region=-r_rod_60pct_down_f&+z_bottom_reflector_60pct_down_f&-z_bottom_absorber_60pct_down_f)
cr_pipe_60pct_down_c = openmc.Cell(fill=t91,region=+r_rod_60pct_down_f&-r_pipe_60pct_down_f&+z_bottom_reflector_60pct_down_f&-z_top_absorber_60pct_down_f)
cr_coolant_60pct_down_c = openmc.Cell(fill=coolant,region=+r_pipe_60pct_down_f|-z_bottom_reflector_60pct_down_f|+z_top_absorber_60pct_down_f)

cr_60pct_down_u = openmc.Universe(cells=[cr_absorber_60pct_down_c,cr_reflector_60pct_down_c,cr_pipe_60pct_down_c,cr_coolant_60pct_down_c])

# 60pct control rod universe up
r_rod_60pct_up_f = openmc.ZCylinder(r=2.0)
r_pipe_60pct_up_f = openmc.ZCylinder(r=2.2)
z_bottom_absorber_60pct_up_f = openmc.ZPlane(z0=91)
z_top_absorber_60pct_up_f = openmc.ZPlane(z0=183)
z_bottom_reflector_60pct_up_f = openmc.ZPlane(z0=-1)

cr_absorber_60pct_up_c = openmc.Cell(fill=cr_60pct_b4c,region=-r_rod_60pct_up_f&+z_bottom_absorber_60pct_up_f&-z_top_absorber_60pct_up_f)
cr_reflector_60pct_up_c = openmc.Cell(fill=yzro,region=-r_rod_60pct_up_f&+z_bottom_reflector_60pct_up_f&-z_bottom_absorber_60pct_up_f)
cr_pipe_60pct_up_c = openmc.Cell(fill=t91,region=+r_rod_60pct_up_f&-r_pipe_60pct_up_f&+z_bottom_reflector_60pct_up_f&-z_top_absorber_60pct_up_f)
cr_coolant_60pct_up_c = openmc.Cell(fill=coolant,region=+r_pipe_60pct_up_f|-z_bottom_reflector_60pct_up_f|+z_top_absorber_60pct_up_f)

cr_60pct_up_u = openmc.Universe(cells=[cr_absorber_60pct_up_c,cr_reflector_60pct_up_c,cr_pipe_60pct_up_c,cr_coolant_60pct_up_c])

# 90pct safety rod universe down
r_rod_90pct_down_f = openmc.ZCylinder(r=2.0)
r_pipe_90pct_down_f = openmc.ZCylinder(r=2.2)
z_bottom_absorber_90pct_down_f = openmc.ZPlane(z0=-1)
z_top_absorber_90pct_down_f = openmc.ZPlane(z0=91)
z_bottom_reflector_90pct_down_f = openmc.ZPlane(z0=-93)

sr_absorber_90pct_down_c = openmc.Cell(fill=sr_90pct_b4c,region=-r_rod_90pct_down_f&+z_bottom_absorber_90pct_down_f&-z_top_absorber_90pct_down_f)
sr_reflector_90pct_down_c = openmc.Cell(fill=yzro,region=-r_rod_90pct_down_f&+z_bottom_reflector_90pct_down_f&-z_bottom_absorber_90pct_down_f)
sr_pipe_90pct_down_c = openmc.Cell(fill=t91,region=+r_rod_90pct_down_f&-r_pipe_90pct_down_f&+z_bottom_reflector_90pct_down_f&-z_top_absorber_90pct_down_f)
sr_coolant_90pct_down_c = openmc.Cell(fill=coolant,region=+r_pipe_90pct_down_f|-z_bottom_reflector_90pct_down_f|+z_top_absorber_90pct_down_f)

sr_90pct_down_u = openmc.Universe(cells=[sr_absorber_90pct_down_c,sr_reflector_90pct_down_c,sr_pipe_90pct_down_c,sr_coolant_90pct_down_c])

# 90pct safety rod universe up
r_rod_90pct_up_f = openmc.ZCylinder(r=2.0)
r_pipe_90pct_up_f = openmc.ZCylinder(r=2.2)
z_bottom_absorber_90pct_up_f = openmc.ZPlane(z0=91)
z_top_absorber_90pct_up_f = openmc.ZPlane(z0=183)
z_bottom_reflector_90pct_up_f = openmc.ZPlane(z0=-1)

sr_absorber_90pct_up_c = openmc.Cell(fill=sr_90pct_b4c,region=-r_rod_90pct_up_f&+z_bottom_absorber_90pct_up_f&-z_top_absorber_90pct_up_f)
sr_reflector_90pct_up_c = openmc.Cell(fill=yzro,region=-r_rod_90pct_up_f&+z_bottom_reflector_90pct_up_f&-z_bottom_absorber_90pct_up_f)
sr_pipe_90pct_up_c = openmc.Cell(fill=t91,region=+r_rod_90pct_up_f&-r_pipe_90pct_up_f&+z_bottom_reflector_90pct_up_f&-z_top_absorber_90pct_up_f)
sr_coolant_90pct_up_c = openmc.Cell(fill=coolant,region=+r_pipe_90pct_up_f|-z_bottom_reflector_90pct_up_f|+z_top_absorber_90pct_up_f)

sr_90pct_up_u = openmc.Universe(cells=[sr_absorber_90pct_up_c,sr_reflector_90pct_up_c,sr_pipe_90pct_up_c,sr_coolant_90pct_up_c])

# reflect pin universe
reflect_pin_c = openmc.Cell(fill=yzro,region=-r_protect_gas_f&+z_bottom_lower_plug_f&-z_top_upper_plug_f)
reflect_cladding_c = openmc.Cell(fill=t91,region=-r_cladding_f&+r_protect_gas_f&+z_bottom_lower_plug_f&-z_top_upper_plug_f)
reflect_coolant_c = openmc.Cell(fill=coolant,region=+r_cladding_f|-z_bottom_lower_plug_f|+z_top_upper_plug_f)

reflect_pin_u = openmc.Universe(cells=[reflect_pin_c,reflect_cladding_c,reflect_coolant_c])

# coolant universe for empty pin space filling
Pb_Bi_coolant_cell = openmc.Cell(fill=coolant)
Pb_Bi_coolant_u = openmc.Universe(cells=[Pb_Bi_coolant_cell])

Geometry specification of each kind of assembly universe

In [4]:
duct_inner_surface = openmc.model.hexagonal_prism(edge_length=11.83568,orientation='x')
hex_outer_thimble = openmc.model.hexagonal_prism(edge_length=3.2331615,orientation='x')
hex_inner_thimble = openmc.model.hexagonal_prism(edge_length=3.0022214,orientation='x')

inner_lat = openmc.HexLattice(name="inner assembly")
inner_lat.center = (0., 0.)
inner_lat.pitch = (1.36,)
inner_lat.orientation = 'x'
inner_lat.outer = Pb_Bi_coolant_u


# inner fuel assembly with 90pct SR
inner_s_lat = openmc.HexLattice(name="inner_s assembly")
inner_s_lat.center = (0., 0.)
inner_s_lat.pitch = (1.36,)
inner_s_lat.orientation = 'x'
inner_s_lat.outer = Pb_Bi_coolant_u

inone = [Pb_Bi_coolant_u]*1
intwo = [Pb_Bi_coolant_u]*6
inthree = [Pb_Bi_coolant_u]*12
infour = [inner_pin_u]*18
infive = [inner_pin_u]*24
insix = [inner_pin_u]*30
inseven = [inner_pin_u]*36
ineight = [inner_pin_u]*42
innine = [inner_pin_u]*48
inner_s_lat.universes = [innine,ineight,inseven,insix,infive,infour,inthree,intwo,inone]
inner_s_absorber_c = openmc.Cell(fill=sr_90pct_up_u,region=hex_inner_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
inner_s_thimble_c = openmc.Cell(fill=t91,region=~hex_inner_thimble&hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
inner_s_lat_c = openmc.Cell(fill=inner_s_lat,region=duct_inner_surface&~hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
inner_s_lat_coolant_c = openmc.Cell(fill=coolant,region=~duct_inner_surface|+z_top_upper_plug_f|-z_bottom_lower_plug_f)

inner_s_assembly_u = openmc.Universe(cells=[inner_s_absorber_c,inner_s_lat_c,inner_s_lat_coolant_c,inner_s_thimble_c])


# inner fuel assembly with 40pct CR
inner_c_lat = openmc.HexLattice(name="inner_c assembly")
inner_c_lat.center = (0., 0.)
inner_c_lat.pitch = (1.36,)
inner_c_lat.orientation = 'x'
inner_c_lat.outer = Pb_Bi_coolant_u

inone = [Pb_Bi_coolant_u]*1
intwo = [Pb_Bi_coolant_u]*6
inthree = [Pb_Bi_coolant_u]*12
infour = [inner_pin_u]*18
infive = [inner_pin_u]*24
insix = [inner_pin_u]*30
inseven = [inner_pin_u]*36
ineight = [inner_pin_u]*42
innine = [inner_pin_u]*48
inner_c_lat.universes = [innine,ineight,inseven,insix,infive,infour,inthree,intwo,inone]
inner_c_absorber_c = openmc.Cell(fill=cr_40pct_down_u,region=hex_inner_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
inner_c_thimble_c = openmc.Cell(fill=t91,region=~hex_inner_thimble&hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
inner_c_lat_c = openmc.Cell(fill=inner_c_lat,region=duct_inner_surface&~hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
inner_c_lat_coolant_c = openmc.Cell(fill=coolant,region=~duct_inner_surface|+z_top_upper_plug_f|-z_bottom_lower_plug_f)

inner_c_assembly_u = openmc.Universe(cells=[inner_c_absorber_c,inner_c_lat_c,inner_c_lat_coolant_c,inner_c_thimble_c])


# middle fuel assembly with 90pct SR
middle_s_lat = openmc.HexLattice(name="middle_s assembly")
middle_s_lat.center = (0., 0.)
middle_s_lat.pitch = (1.36,)
middle_s_lat.orientation = 'x'
middle_s_lat.outer = Pb_Bi_coolant_u

inone = [Pb_Bi_coolant_u]*1
intwo = [Pb_Bi_coolant_u]*6
inthree = [Pb_Bi_coolant_u]*12
infour = [middle_pin_u]*18
infive = [middle_pin_u]*24
insix = [middle_pin_u]*30
inseven = [middle_pin_u]*36
ineight = [middle_pin_u]*42
innine = [middle_pin_u]*48
middle_s_lat.universes = [innine,ineight,inseven,insix,infive,infour,inthree,intwo,inone]
middle_s_absorber_c = openmc.Cell(fill=sr_90pct_up_u,region=hex_inner_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
middle_s_thimble_c = openmc.Cell(fill=t91,region=~hex_inner_thimble&hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
middle_s_lat_c = openmc.Cell(fill=middle_s_lat,region=duct_inner_surface&~hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
middle_s_lat_coolant_c = openmc.Cell(fill=coolant,region=~duct_inner_surface|+z_top_upper_plug_f|-z_bottom_lower_plug_f)

middle_s_assembly_u = openmc.Universe(cells=[middle_s_absorber_c,middle_s_lat_c,middle_s_lat_coolant_c,middle_s_thimble_c])


# middle fuel assembly with 40pct CR
middle_c_lat = openmc.HexLattice(name="middle_c assembly")
middle_c_lat.center = (0., 0.)
middle_c_lat.pitch = (1.36,)
middle_c_lat.orientation = 'x'
middle_c_lat.outer = Pb_Bi_coolant_u

inone = [Pb_Bi_coolant_u]*1
intwo = [Pb_Bi_coolant_u]*6
inthree = [Pb_Bi_coolant_u]*12
infour = [middle_pin_u]*18
infive = [middle_pin_u]*24
insix = [middle_pin_u]*30
inseven = [middle_pin_u]*36
ineight = [middle_pin_u]*42
innine = [middle_pin_u]*48
middle_c_lat.universes = [innine,ineight,inseven,insix,infive,infour,inthree,intwo,inone]
middle_c_absorber_c = openmc.Cell(fill=cr_40pct_down_u,region=hex_inner_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
middle_c_thimble_c = openmc.Cell(fill=t91,region=~hex_inner_thimble&hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
middle_c_lat_c = openmc.Cell(fill=middle_c_lat,region=duct_inner_surface&~hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
middle_c_lat_coolant_c = openmc.Cell(fill=coolant,region=~duct_inner_surface|+z_top_upper_plug_f|-z_bottom_lower_plug_f)

middle_c_assembly_u = openmc.Universe(cells=[middle_c_absorber_c,middle_c_lat_c,middle_c_lat_coolant_c,middle_c_thimble_c])


# outer fuel assembly with 90pct SR
outer_s_lat = openmc.HexLattice(name="outer_s assembly")
outer_s_lat.center = (0., 0.)
outer_s_lat.pitch = (1.36,)
outer_s_lat.orientation = 'x'
outer_s_lat.outer = Pb_Bi_coolant_u

inone = [Pb_Bi_coolant_u]*1
intwo = [Pb_Bi_coolant_u]*6
inthree = [Pb_Bi_coolant_u]*12
infour = [outer_pin_u]*18
infive = [outer_pin_u]*24
insix = [outer_pin_u]*30
inseven = [outer_pin_u]*36
ineight = [outer_pin_u]*42
innine = [outer_pin_u]*48
outer_s_lat.universes = [innine,ineight,inseven,insix,infive,infour,inthree,intwo,inone]
outer_s_absorber_c = openmc.Cell(fill=sr_90pct_up_u,region=hex_inner_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
outer_s_thimble_c = openmc.Cell(fill=t91,region=~hex_inner_thimble&hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
outer_s_lat_c = openmc.Cell(fill=outer_s_lat,region=duct_inner_surface&~hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
outer_s_lat_coolant_c = openmc.Cell(fill=coolant,region=~duct_inner_surface|+z_top_upper_plug_f|-z_bottom_lower_plug_f)

outer_s_assembly_u = openmc.Universe(cells=[outer_s_absorber_c,outer_s_lat_c,outer_s_lat_coolant_c,outer_s_thimble_c])


# outer fuel assembly with 40pct CR
outer_c_lat = openmc.HexLattice(name="outer_c assembly")
outer_c_lat.center = (0., 0.)
outer_c_lat.pitch = (1.36,)
outer_c_lat.orientation = 'x'
outer_c_lat.outer = Pb_Bi_coolant_u

inone = [Pb_Bi_coolant_u]*1
intwo = [Pb_Bi_coolant_u]*6
inthree = [Pb_Bi_coolant_u]*12
infour = [outer_pin_u]*18
infive = [outer_pin_u]*24
insix = [outer_pin_u]*30
inseven = [outer_pin_u]*36
ineight = [outer_pin_u]*42
innine = [outer_pin_u]*48
outer_c_lat.universes = [innine,ineight,inseven,insix,infive,infour,inthree,intwo,inone]
outer_c_absorber_c = openmc.Cell(fill=cr_40pct_down_u,region=hex_inner_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
outer_c_thimble_c = openmc.Cell(fill=t91,region=~hex_inner_thimble&hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
outer_c_lat_c = openmc.Cell(fill=outer_c_lat,region=duct_inner_surface&~hex_outer_thimble&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
outer_c_lat_coolant_c = openmc.Cell(fill=coolant,region=~duct_inner_surface|+z_top_upper_plug_f|-z_bottom_lower_plug_f)

outer_c_assembly_u = openmc.Universe(cells=[outer_c_absorber_c,outer_c_lat_c,outer_c_lat_coolant_c,outer_c_thimble_c])


# reflect assembly
reflect_lat = openmc.HexLattice(name="reflect assembly")
reflect_lat.center = (0., 0.)
reflect_lat.pitch = (1.36,)
reflect_lat.orientation = 'x'
reflect_lat.outer = Pb_Bi_coolant_u

inone = [reflect_pin_u]*1
intwo = [reflect_pin_u]*6
inthree = [reflect_pin_u]*12
infour = [reflect_pin_u]*18
infive = [reflect_pin_u]*24
insix = [reflect_pin_u]*30
inseven = [reflect_pin_u]*36
ineight = [reflect_pin_u]*42
innine = [reflect_pin_u]*48
reflect_lat.universes = [innine,ineight,inseven,insix,infive,infour,inthree,intwo,inone]
reflect_lat_c = openmc.Cell(fill=reflect_lat,region=duct_inner_surface&-z_top_upper_plug_f&+z_bottom_lower_plug_f)
reflect_lat_coolant_c = openmc.Cell(fill=coolant,region=~duct_inner_surface|+z_top_upper_plug_f|-z_bottom_lower_plug_f)

reflect_assembly_u = openmc.Universe(cells=[reflect_lat_c,reflect_lat_coolant_c])

/home/ssn/miniconda3/envs/openmc-new/lib/python3.12/site-packages/openmc/model/funcs.py:124: FutureWarning: The hexagonal_prism(...) function has been replaced by the HexagonalPrism(...) class. Future versions of OpenMC will not accept hexagonal_prism.
  warn("The hexagonal_prism(...) function has been replaced by the "


Geometry specification of whole core

In [5]:
outer_core_surface = openmc.ZCylinder(r=125, boundary_type='vacuum')
top_core_surface = openmc.ZPlane(z0=200, boundary_type='vacuum')
bottom_core_surface = openmc.ZPlane(z0=-50, boundary_type='vacuum')

core_lat = openmc.HexLattice(name='core')
core_lat.center = (0., 0.)
core_lat.pitch = (20.5,)
core_lat.outer = Pb_Bi_coolant_u

one = [inner_s_assembly_u]*1
two = ([inner_c_assembly_u]*1+[inner_s_assembly_u]*1)*3
three = ([middle_c_assembly_u]*1+[middle_s_assembly_u]*1)*6
four = ([outer_s_assembly_u]*1+[outer_c_assembly_u]*2)*6
five = [reflect_assembly_u]*24
core_lat.universes = [five,four,three,two,one]

rotation_angle=(0.,0.,30.)
core = openmc.Cell(fill=core_lat,region=-outer_core_surface & -top_core_surface & +bottom_core_surface)
core.rotation=rotation_angle
geom = openmc.Geometry([core])
geom.export_to_xml()

energy_filter=openmc.EnergyFilter([0,2.5E-02,1.0E-01,1.0,10,
                                   1.0E+02,2.0E+02,3.0E+02,4.0E+02,5.0E+02,6.0E+02,7.0E+02,8.0E+02,9.0E+02,
                                   1.0E+03,2.0E+03,3.0E+03,4.0E+03,5.0E+03,6.0E+03,7.0E+03,8.0E+03,9.0E+03,
                                   1.0E+04,1.5E+04,2.0E+04,2.5E+04,3.0E+04,3.5E+04,4.0E+04,4.5E+04,5.0E+04,
                                   5.5E+04,6.0E+04,6.5E+04,7.0E+04,7.5E+04,8.0E+04,8.5E+04,9.0E+04,9.5E+04,
                                   1.0E+05,1.5E+05,2.0E+05,2.5E+05,3.0E+05,3.5E+05,4.0E+05,4.5E+05,5.0E+05,
                                   5.5E+05,6.0E+05,6.5E+05,7.0E+05,7.5E+05,8.0E+05,8.5E+05,9.0E+05,9.5E+05,
                                   1.0E+06,2.0E+06,4.0E+06,5.0E+06,6.0E+06,7.0E+06,8.0E+06,9.0E+06,1.0E+07,2.0E+07,5.0E+07])
tally1 = openmc.Tally()
tally1.filters = [openmc.DistribcellFilter(inner_s_lat_c)]
tally1.scores = ['fission-q-recoverable','fission-q-prompt','nu-fission','prompt-nu-fission','fission','flux']
tally2 = openmc.Tally()
tally2.filters = [openmc.DistribcellFilter(inner_c_lat_c)]
tally2.scores = ['fission-q-recoverable','fission-q-prompt','nu-fission','prompt-nu-fission','fission','flux']
tally3 = openmc.Tally()
tally3.filters = [openmc.DistribcellFilter(middle_s_lat_c)]
tally3.scores = ['fission-q-recoverable','fission-q-prompt','nu-fission','prompt-nu-fission','fission','flux']
tally4 = openmc.Tally()
tally4.filters = [openmc.DistribcellFilter(middle_c_lat_c)]
tally4.scores = ['fission-q-recoverable','fission-q-prompt','nu-fission','prompt-nu-fission','fission','flux']
tally5 = openmc.Tally()
tally5.filters = [openmc.DistribcellFilter(outer_s_lat_c)]
tally5.scores = ['fission-q-recoverable','fission-q-prompt','nu-fission','prompt-nu-fission','fission','flux']
tally6 = openmc.Tally()
tally6.filters = [openmc.DistribcellFilter(outer_c_lat_c)]
tally6.scores = ['fission-q-recoverable','fission-q-prompt','nu-fission','prompt-nu-fission','fission','flux']

tally11 = openmc.Tally()
tally11.filters = [openmc.CellFilter(inner_s_lat_c),energy_filter]
tally11.scores = ['flux']
tally12 = openmc.Tally()
tally12.filters = [openmc.CellFilter(inner_c_lat_c),energy_filter]
tally12.scores = ['flux']
tally13 = openmc.Tally()
tally13.filters = [openmc.CellFilter(middle_s_lat_c),energy_filter]
tally13.scores = ['flux']
tally14 = openmc.Tally()
tally14.filters = [openmc.CellFilter(middle_c_lat_c),energy_filter]
tally14.scores = ['flux']
tally15 = openmc.Tally()
tally15.filters = [openmc.CellFilter(outer_s_lat_c),energy_filter]
tally15.scores = ['flux']
tally16 = openmc.Tally()
tally16.filters = [openmc.CellFilter(outer_c_lat_c),energy_filter]
tally16.scores = ['flux']

groups = mgxs.EnergyGroups()
groups.group_edges = np.array([0,1e8])
core_beta = mgxs.Beta(domain_type='cell',domain=core,energy_groups=groups)

tallies = openmc.Tallies([tally1,tally2,tally3,tally4,tally5,tally6,tally11,tally12,tally13,tally14,tally15,tally16])
tallies +=core_beta.tallies.values()
tallies.export_to_xml()

In [6]:
tally1 = openmc.Tally()
tally1.filters = [openmc.DistribcellFilter(inner_s_lat_c)]
tally1.scores = ['heating','fission-q-recoverable','fission-q-prompt']
tally2 = openmc.Tally()
tally2.filters = [openmc.DistribcellFilter(inner_c_lat_c)]
tally2.scores = ['heating','fission-q-recoverable','fission-q-prompt']
tally3 = openmc.Tally()
tally3.filters = [openmc.DistribcellFilter(middle_s_lat_c)]
tally3.scores = ['heating','fission-q-recoverable','fission-q-prompt']
tally4 = openmc.Tally()
tally4.filters = [openmc.DistribcellFilter(middle_c_lat_c)]
tally4.scores = ['heating','fission-q-recoverable','fission-q-prompt']
tally5 = openmc.Tally()
tally5.filters = [openmc.DistribcellFilter(outer_s_lat_c)]
tally5.scores = ['heating','fission-q-recoverable','fission-q-prompt']
tally6 = openmc.Tally()
tally6.filters = [openmc.DistribcellFilter(outer_c_lat_c)]
tally6.scores = ['heating','fission-q-recoverable','fission-q-prompt']

tally11 = openmc.Tally()
tally11.filters = [openmc.DistribcellFilter(inner_active_pin_c)]
tally11.scores = ['heating','fission-q-recoverable','fission-q-prompt']
tally12 = openmc.Tally()
tally12.filters = [openmc.DistribcellFilter(middle_active_pin_c)]
tally12.scores = ['heating','fission-q-recoverable','fission-q-prompt']
tally13 = openmc.Tally()
tally13.filters = [openmc.DistribcellFilter(outer_active_pin_c)]
tally13.scores = ['heating','fission-q-recoverable','fission-q-prompt']


groups = mgxs.EnergyGroups(group_edges = np.array([0,1e8]))
# groups.group_edges = np.array([0,1e8])
core_beta = mgxs.Beta(domain_type='cell',domain=core,energy_groups=groups)

tallies = openmc.Tallies([tally1,tally2,tally3,tally4,tally5,tally6,tally11,tally12,tally13])
tallies +=core_beta.tallies.values()
tallies.export_to_xml()

energy_group_structure=openmc.mgxs.GROUP_STRUCTURES['ECCO-1968']
energy_filter=openmc.EnergyFilter(energy_group_structure)
fuel_filter=openmc.CellFilter([inner_active_pin_c.id,middle_active_pin_c.id,outer_active_pin_c.id])
core_filter=openmc.CellFilter(core)

tally_material_spectrum=openmc.Tally()
tally_material_spectrum.filters=[fuel_filter,energy_filter]
tally_material_spectrum.scores=['flux']

tally_material_heating=openmc.Tally()
tally_material_heating.filters=[fuel_filter]
tally_material_heating.scores=['heating']

tally_cell_spectrum=openmc.Tally()
tally_cell_spectrum.filters=[core_filter,energy_filter]
tally_cell_spectrum.scores=['flux']

tally_cell_heating=openmc.Tally()
tally_cell_heating.filters=[core_filter]
tally_cell_heating.scores=['heating']

tally_fission1_reactions=openmc.Tally()
tally_fission1_reactions.filters=[core_filter,energy_filter]
tally_fission1_reactions.scores=['fission']
tally_fission1_reactions.nuclides=['U235']

tally_fission2_reactions=openmc.Tally()
tally_fission2_reactions.filters=[core_filter,energy_filter]
tally_fission2_reactions.scores=['fission']
tally_fission2_reactions.nuclides=['U238']

# tally_fission3_reactions=openmc.Tally()
# tally_fission3_reactions.filters=[core_filter,energy_filter]
# tally_fission3_reactions.scores=['fission']
# tally_fission3_reactions.nuclides=['Pu239']

tallies = openmc.Tallies([tally_material_spectrum,tally_material_heating,tally_cell_spectrum,,tally_fission1_reactions,tally_fission2_reactions])
tallies.export_to_xml()tally_cell_heating

In [7]:
# mesh to calculate shannon entropy 
m = openmc.RegularMesh()
m.lower_left, m.upper_right = geom.bounding_box
m.dimension = (20, 20, 15)

# uniform sidtribution of initial guess source spots
lower_left = [-72, -72, 0]
upper_right = [72, 72, 90]
uniform_dist = openmc.stats.Box(lower_left, upper_right, only_fissionable=True)
src = openmc.IndependentSource(space=uniform_dist)

# eigenvalue problem setting
settings = openmc.Settings()
settings.source = src
settings.batches = 200
settings.inactive = 20
settings.particles = 40000
# shannon entropy setting
settings.entropy_mesh = m

settings.export_to_xml()

In [8]:
openmc.run()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

 Reading U238 from /home/ssn/miniconda3/endfb80/U238.h5
 Reading He3 from /home/ssn/miniconda3/endfb80/He3.h5
 Reading He4 from /home/ssn/miniconda3/endfb80/He4.h5
 Reading C12 from /home/ssn/miniconda3/endfb80/C12.h5
 Reading C13 from /home/ssn/miniconda3/endfb80/C13.h5
 Reading Cr50 from /home/ssn/miniconda3/endfb80/Cr50.h5
 Reading Cr52 from /home/ssn/miniconda3/endfb80/Cr52.h5
 Reading Cr53 from /home/ssn/miniconda3/endfb80/Cr53.h5
 Reading Cr54 from /home/ssn/miniconda3/endfb80/Cr54.h5
 Reading Ni58 from /home/ssn/miniconda3/endfb80/Ni58.h5
 Reading Ni60 from /home/ssn/miniconda3/endfb80/Ni60.h5
 Reading Ni61 from /home/ssn/miniconda3/endfb80/Ni61.h5
 Reading Ni62 from /home/ssn/miniconda3/endfb80/Ni62.h5
 Reading Ni64 from /home/ssn/miniconda3/endfb80/Ni64.h5
 Reading Mn55 from /home/ssn/miniconda3/endfb80/Mn55.h5
 Reading Mo92 from /home/ssn/miniconda3/endfb80/Mo92.h5
 Reading Mo94 from /home/ssn/miniconda3/endfb80/Mo94.h5
 Reading Mo95 from /home/ssn/miniconda3/endfb80/Mo95.h5
